# 00 – Kaggle environment setup (Thành viên B – nhánh Học sâu)

Notebook này chỉ làm 4 việc, đúng quy tắc "Kaggle Notebook chỉ clone + cài + gọi script":
1. Kiểm tra GPU (cần **Accelerator = GPU T4 x2**, **Internet = On** trong Settings).
2. Cài `open_clip_torch`, `faiss-gpu`, `scikit-image`.
3. `git clone` repo và kiểm tra import package `sbir`.
4. Chạy sanity check `PromptedCLIP` bằng dummy tensor (không cần dataset của Thành viên A) + smoke test FAISS GPU.

In [ ]:
!nvidia-smi
import sys, platform
print(sys.version)

## 1. Cài thư viện
`faiss-gpu` bản pip cũ không còn hỗ trợ Python ≥ 3.11, nên dùng `faiss-gpu-cu12`; nếu lỗi sẽ tự lùi về `faiss-cpu` (vẫn chạy được, chỉ chậm hơn).

In [ ]:
!pip install -q open_clip_torch scikit-image
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu-cu12"], capture_output=True, text=True)
if r.returncode != 0:
    print("faiss-gpu-cu12 failed, falling back to faiss-cpu
", r.stderr[-500:])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)

## 2. Clone repo
Nếu repo để **private**: thêm Kaggle Secret tên `GITHUB_TOKEN` (Add-ons → Secrets) rồi chạy lại cell.

In [ ]:
import os
REPO = "github.com/thaihoang104569/CS406.git"
BRANCH = "main"
WORK = "/kaggle/working/CS406"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.isdir(WORK):
    !git -C {WORK} pull -q
else:
    !git clone -q -b {BRANCH} {url} {WORK}
!git -C {WORK} log --oneline -3

os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

## 3. Kiểm tra phiên bản và import

In [ ]:
import torch, open_clip, faiss, skimage
print("torch     ", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
print("open_clip ", open_clip.__version__)
print("faiss     ", faiss.__version__, "| faiss GPUs:", faiss.get_num_gpus() if hasattr(faiss, "get_num_gpus") else 0)
print("skimage   ", skimage.__version__)

from sbir.deep.prompt_clip import PromptedCLIP, build_prompted_clip
from sbir.deep.losses import SBIRTripletLoss, SemanticClassificationLoss
from sbir.index import VectorIndex
print("sbir imports OK")

## 4. Sanity check PromptedCLIP với dummy tensor
Dùng trọng số **OpenAI CLIP ViT-B/32 thật**, batch 64, AMP FP16 – giống cấu hình huấn luyện E3/E4 – để đo VRAM và thời gian/step trước khi có dữ liệu thật. Thử thêm `--num_prompts 1` / `5` nếu muốn.

In [ ]:
!python scripts/sanity_prompt_clip.py --pretrained openai --num_prompts 3 --batch_size 64 --num_classes 8 --steps 20 --amp

## 5. Smoke test FAISS (GPU nếu có)

In [ ]:
import numpy as np, time
use_gpu = hasattr(faiss, "get_num_gpus") and faiss.get_num_gpus() > 0
rng = np.random.default_rng(0)
gallery = rng.standard_normal((17000, 512)).astype("float32")   # ~ cỡ gallery test Sketchy
queries = rng.standard_normal((15000, 512)).astype("float32")
idx = VectorIndex(dim=512, use_gpu=use_gpu)
idx.add(gallery)
t = time.time(); scores, ids = idx.search(queries, top_k=200); dt = time.time() - t
print(f"use_gpu={use_gpu}  scores={scores.shape}  {dt:.2f}s for {len(queries)} queries")
# kết quả FAISS phải trùng với nhân ma trận NumPy
g = gallery / np.linalg.norm(gallery, axis=1, keepdims=True)
q = queries[:5] / np.linalg.norm(queries[:5], axis=1, keepdims=True)
assert np.allclose(np.sort(-(q @ g.T), axis=1)[:, :200], -scores[:5], atol=1e-4)
print("FAISS OK")